In [1]:
!pip install qiskit
!pip install qiskit-ibm-runtime
!pip -q install -U qiskit qiskit-aer qiskit-ibm-runtime pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.6/412.6 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.7/120.7 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.2/224.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.6/76.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.5/541.5 kB 39.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/1

In [ ]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService(
    channel="ibm_cloud",
    token="",
    instance=""
)
backend = service.least_busy(operational=True, simulator=False)

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
# result = sampler_job.result()

# job_result = sampler_job.result()
# circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
# circuit_schedule_timing = circuit_schedule["timing"]

import plotly.io as pio
pio.renderers.default = "colab"

job_result = sampler_job.result()

print("Job status:", sampler_job.status())
print("Job ID:", sampler_job.job_id())
print("Metadata keys:", job_result[0].metadata.keys())

circuit_schedule_timing = job_result[0].metadata["compilation"]["scheduler_timing"]["timing"]

from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

fig.show(renderer="colab")

qiskit_runtime_service._discover_account:WARNING:2026-07-22 20:47:38,501: Loading account with the given token. A saved account will not be used.


KeyboardInterrupt: 

In [ ]:
import math
import pandas as pd
import qiskit_ibm_runtime.fake_provider as fp

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator


SHOTS = 200
MIN_QUBITS = 2
MAX_QUBITS = 10
OPTIMIZATION_LEVEL = 3


DEVICE_CLASSES = {
    "ibm_boston": "FakeBoston",
    "ibm_marrakesh": "FakeMarrakesh",
    "ibm_kingston": "FakeKingston",
    "ibm_pittsburg": "FakePittsburgh",
    "ibm_fez": "FakeFez",
}


def make_ghz(n):
    qc = QuantumCircuit(n, n)
    qc.h(0)

    for i in range(n - 1):
        qc.cx(i, i + 1)

    qc.measure(range(n), range(n))
    return qc


def ghz_support_fidelity(counts, n, shots):
    zero = "0" * n
    one = "1" * n

    cleaned = {}
    for key, value in counts.items():
        k = key.replace(" ", "")
        cleaned[k] = cleaned.get(k, 0) + value

    p_zero = cleaned.get(zero, 0) / shots
    p_one = cleaned.get(one, 0) / shots

    return p_zero + p_one, p_zero, p_one


rows = []

for device_name, class_name in DEVICE_CLASSES.items():
    print(f"\nLoading {device_name} using {class_name}")

    try:
        BackendClass = getattr(fp, class_name)
        backend = BackendClass()
    except Exception as e:
        print(f"  Could not instantiate {class_name}: {repr(e)}")
        rows.append({
            "device": device_name,
            "fake_backend_class": class_name,
            "ghz_qubits": None,
            "shots": SHOTS,
            "fidelity": None,
            "p_zero": None,
            "p_one": None,
            "depth": None,
            "ops": None,
            "status": "backend_class_not_found",
            "error": repr(e),
        })
        continue

    simulator = AerSimulator.from_backend(backend)
    simulator.set_options(seed_simulator=1234)

    for n in range(MIN_QUBITS, MAX_QUBITS + 1):
        print(f"  GHZ({n})...", end=" ")

        try:
            qc = make_ghz(n)

            tqc = transpile(
                qc,
                backend=backend,
                optimization_level=OPTIMIZATION_LEVEL,
                seed_transpiler=1234,
            )

            result = simulator.run(tqc, shots=SHOTS).result()
            counts = result.get_counts()

            fidelity, p_zero, p_one = ghz_support_fidelity(counts, n, SHOTS)

            rows.append({
                "device": device_name,
                "fake_backend_class": class_name,
                "backend_name": backend.name,
                "ghz_qubits": n,
                "shots": SHOTS,
                "fidelity": fidelity,
                "p_zero": p_zero,
                "p_one": p_one,
                "depth": tqc.depth(),
                "ops": dict(tqc.count_ops()),
                "status": "ok",
                "error": "",
            })

            print(
                f"fidelity={fidelity:.4f}, "
                f"p0={p_zero:.4f}, "
                f"p1={p_one:.4f}, "
                f"depth={tqc.depth()}"
            )

        except Exception as e:
            print(f"failed: {repr(e)}")

            rows.append({
                "device": device_name,
                "fake_backend_class": class_name,
                "backend_name": getattr(backend, "name", None),
                "ghz_qubits": n,
                "shots": SHOTS,
                "fidelity": None,
                "p_zero": None,
                "p_one": None,
                "depth": None,
                "ops": None,
                "status": "failed",
                "error": repr(e),
            })


df = pd.DataFrame(rows)
df


Loading ibm_boston using FakeBoston
  GHZ(2)... fidelity=0.9900, p0=0.5200, p1=0.4700, depth=7
  GHZ(3)... fidelity=0.9900, p0=0.5200, p1=0.4700, depth=10
  GHZ(4)... fidelity=0.9950, p0=0.5250, p1=0.4700, depth=13
  GHZ(5)... fidelity=0.9750, p0=0.5150, p1=0.4600, depth=16
  GHZ(6)... fidelity=0.9700, p0=0.5050, p1=0.4650, depth=19
  GHZ(7)... fidelity=0.9700, p0=0.5100, p1=0.4600, depth=22
  GHZ(8)... fidelity=0.9700, p0=0.5050, p1=0.4650, depth=25
  GHZ(9)... fidelity=0.9600, p0=0.4950, p1=0.4650, depth=28
  GHZ(10)... fidelity=0.9600, p0=0.4950, p1=0.4650, depth=31

Loading ibm_marrakesh using FakeMarrakesh
  GHZ(2)... fidelity=0.9850, p0=0.5200, p1=0.4650, depth=7
  GHZ(3)... fidelity=0.9800, p0=0.5200, p1=0.4600, depth=10
  GHZ(4)... fidelity=0.9850, p0=0.5150, p1=0.4700, depth=13
  GHZ(5)... fidelity=0.9550, p0=0.5100, p1=0.4450, depth=16
  GHZ(6)... fidelity=0.9700, p0=0.5050, p1=0.4650, depth=19
  GHZ(7)... fidelity=0.9550, p0=0.5000, p1=0.4550, depth=22
  GHZ(8)... fidelity=

,device,fake_backend_class,backend_name,ghz_qubits,shots,fidelity,p_zero,p_one,depth,ops,status,error
0,ibm_boston,FakeBoston,fake_boston,2,200,0.990,0.520,0.470,7,"{'rz': 5, 'sx': 3, 'measure': 2, 'cz': 1}",ok,
1,ibm_boston,FakeBoston,fake_boston,3,200,0.990,0.520,0.470,10,"{'rz': 8, 'sx': 5, 'measure': 3, 'cz': 2}",ok,
2,ibm_boston,FakeBoston,fake_boston,4,200,0.995,0.525,0.470,13,"{'rz': 11, 'sx': 7, 'measure': 4, 'cz': 3}",ok,
3,ibm_boston,FakeBoston,fake_boston,5,200,0.975,0.515,0.460,16,"{'rz': 14, 'sx': 9, 'measure': 5, 'cz': 4}",ok,
4,ibm_boston,FakeBoston,fake_boston,6,200,0.970,0.505,0.465,19,"{'rz': 17, 'sx': 11, 'measure': 6, 'cz': 5}",ok,
5,ibm_boston,FakeBoston,fake_boston,7,200,0.970,0.510,0.460,22,"{'rz': 20, 'sx': 13, 'measure': 7, 'cz': 6}",ok,
6,ibm_boston,FakeBoston,fake_boston,8,200,0.970,0.505,0.465,25,"{'rz': 23, 'sx': 15, 'measure': 8, 'cz': 7}",ok,
7,ibm_boston,FakeBoston,fake_boston,9,200,0.960,0.495,0.465,28,"{'rz': 26, 'sx': 17, 'measure': 9, 'cz': 8}",ok,
8,ibm_boston,FakeBoston,fake_boston,10,200,0.960,0.495,0.465,31,"{'rz': 29, 'sx': 19, 'measure': 10, 'cz': 9}",ok,
9,ibm_marrakesh,FakeMarrakesh,fake_marrakesh,2,200,0.985,0.520,0.465,7,"{'rz': 5, 'sx': 3, 'measure': 2, 'cz': 1}",ok,


In [ ]:
import math
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional, Dict, Tuple, List, Any

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import qiskit_ibm_runtime.fake_provider as fp


# -----------------------------
# Config
# -----------------------------

SHOTS = 200
MIN_GHZ = 2
MAX_GHZ = 10
OPTIMIZATION_LEVEL = 3
SEED_TRANSPILER = 1234
SEED_SIMULATOR = 1234

DEVICE_CLASSES = {
    "ibm_boston": "FakeBoston",
    "ibm_marrakesh": "FakeMarrakesh",
    "ibm_kingston": "FakeKingston",
    "ibm_pittsburg": "FakePittsburgh",
    "ibm_fez": "FakeFez",
}


# -----------------------------
# Minimal domain model
# -----------------------------

class ExecutionModel:
    GlobalSerialGates = "GlobalSerialGates"
    ParallelByQubitOrEdge = "ParallelByQubitOrEdge"


@dataclass
class QubitCalibrationMetrics:
    readout_fidelity: Optional[float] = None
    prob_measu0_prep1: Optional[float] = None
    prob_measu1_prep0: Optional[float] = None
    readout_error: Optional[float] = None
    gate_errors: Dict[str, float] = field(default_factory=dict)
    gate_durations_ns: Dict[str, int] = field(default_factory=dict)
    t1_seconds: Optional[float] = None
    t2_seconds: Optional[float] = None
    one_qubit_fidelity: Optional[float] = None


@dataclass
class EdgeCalibrationMetrics:
    gate_errors: Dict[str, float] = field(default_factory=dict)
    gate_durations_ns: Dict[str, int] = field(default_factory=dict)
    gate_fidelities: Dict[str, float] = field(default_factory=dict)


@dataclass
class IBMCalibration:
    qubits: List[int]
    edges: List[Tuple[int, int]]
    qubit_metrics: Dict[int, QubitCalibrationMetrics]
    edge_metrics: Dict[Tuple[int, int], EdgeCalibrationMetrics]

    t1_seconds: List[Tuple[int, float]] = field(default_factory=list)
    t2_seconds: List[Tuple[int, float]] = field(default_factory=list)

    prob_measu0_prep1: float = 0.0
    prob_measu1_prep0: float = 0.0

    id_error: float = 0.0
    rx_error: float = 0.0
    pauli_x_error: float = 0.0
    cz_error: float = 0.0
    rzz_error: float = 0.0

    id_length_ns: int = 0
    single_q_gate_length_ns: int = 0
    two_q_gate_length_ns: int = 0
    cz_gate_length_ns: int = 0
    rzz_gate_length_ns: int = 0
    readout_length_ns: int = 0

    t1_avg_seconds: float = 0.0
    t2_avg_seconds: float = 0.0


@dataclass
class CanonicalCalibration:
    eps1q: Dict[Tuple[int, str], float]
    eps2q: Dict[Tuple[Tuple[int, int], str], float]
    eps1q_avg: Optional[float]
    eps2q_avg: Optional[float]
    readout_fidelity: Dict[int, float]
    readout_fidelity_avg: Optional[float]
    t1: Dict[int, float]
    t2: Dict[int, float]
    t1_avg: Optional[float]
    t2_avg: Optional[float]
    dur1q_ns: Dict[str, int]
    dur2q_ns: Dict[str, int]
    dur_meas_ns: Optional[int]
    dur1q_avg_ns: Optional[int]
    dur2q_avg_ns: Optional[int]
    init_survival_per_qubit: Optional[float] = None
    execution_model: str = ExecutionModel.ParallelByQubitOrEdge

    def readout_fid_for(self, q: int) -> float:
        if q in self.readout_fidelity:
            return self.readout_fidelity[q]
        if self.readout_fidelity_avg is not None:
            return self.readout_fidelity_avg
        return 1.0

    def t2_for(self, q: int) -> Optional[float]:
        if q in self.t2:
            return self.t2[q]
        return self.t2_avg

    def eps_for(self, op) -> float:
        k = op.kind

        if k in {"X", "H"}:
            return self.eps1q.get((op.qubits[0], k), self.eps1q_avg or 0.0)

        if k == "RX":
            return self.eps1q.get((op.qubits[0], "Rotate"), self.eps1q_avg or 0.0)

        if k == "RZ":
            return self.eps1q.get((op.qubits[0], "RZ"), self.eps1q_avg or 0.0)

        if k in {"CX", "CZ", "SWAP"}:
            edge = edge_key(op.qubits[0], op.qubits[1])
            return self.eps2q.get((edge, k), self.eps2q_avg or 0.0)

        if k == "CRZ":
            edge = edge_key(op.qubits[0], op.qubits[1])
            return self.eps2q.get((edge, "CRotate"), self.eps2q_avg or 0.0)

        return 0.0

    def duration_ns_for(self, op) -> int:
        k = op.kind

        if k in {"X", "H", "SX", "RZ"}:
            return int(self.dur1q_ns.get(k, self.dur1q_avg_ns or 0))

        if k in {"RX", "RY"}:
            return int(self.dur1q_ns.get("Rotate", self.dur1q_avg_ns or 0))

        if k == "MEASURE":
            return int(self.dur_meas_ns or 0)

        if k in {"CX", "CZ", "SWAP"}:
            return int(self.dur2q_ns.get(k, self.dur2q_avg_ns or 0))

        if k == "CRZ":
            return int(self.dur2q_ns.get("CRotate", self.dur2q_avg_ns or 0))

        return 0


@dataclass
class GateOp:
    kind: str
    qubits: Tuple[int, ...]


@dataclass
class CircuitLike:
    qubits: int
    remaining_gates: List[GateOp]


@dataclass
class FidelityEstimate:
    log_p_ops: float
    log_p_decoh: float
    log_p_total: float
    p_total: float
    per_qubit_end_time_sec: Dict[int, float]


# -----------------------------
# Direct port of helper logic
# -----------------------------

def avg_option(values):
    xs = [float(v) for v in values if v is not None and math.isfinite(float(v))]
    return sum(xs) / len(xs) if xs else None


def pct_to_prob(value):
    if value is None:
        return None
    value = float(value)
    if math.isfinite(value) and value > 0.0:
        return value if value <= 1.0 else value / 100.0
    return None


def seconds_opt(value):
    if value is None:
        return None
    value = float(value)
    if math.isfinite(value) and value > 0.0:
        return value
    return None


def nanos_opt(seconds):
    if seconds is None:
        return None
    seconds = float(seconds)
    if math.isfinite(seconds) and seconds > 0.0:
        return int(seconds * 1e9)
    return None


def avg_long_option(values):
    xs = [int(v) for v in values if v is not None and int(v) > 0]
    return int(sum(xs) / len(xs)) if xs else None


def readout_fidelity(metrics: QubitCalibrationMetrics):
    if metrics.readout_fidelity is not None:
        return metrics.readout_fidelity

    if metrics.prob_measu0_prep1 is not None and metrics.prob_measu1_prep0 is not None:
        return 1.0 - ((metrics.prob_measu0_prep1 + metrics.prob_measu1_prep0) / 2.0)

    if metrics.readout_error is not None:
        return 1.0 - metrics.readout_error

    return None


def upper(name):
    return name.strip().upper()


def edge_key(a, b):
    return (a, b) if a <= b else (b, a)


def metric_error(metrics, *names):
    for name in names:
        key = upper(name)
        if key in metrics.gate_errors:
            return metrics.gate_errors[key]
    return None


def metric_duration(metrics, *names):
    for name in names:
        key = upper(name)
        v = metrics.gate_durations_ns.get(key)
        if v is not None and v > 0:
            return v
    return None


def edge_metric_error(metrics, *names):
    for name in names:
        key = upper(name)
        if key in metrics.gate_errors:
            return metrics.gate_errors[key]
    return None


def edge_metric_duration(metrics, *names):
    for name in names:
        key = upper(name)
        v = metrics.gate_durations_ns.get(key)
        if v is not None and v > 0:
            return v
    return None


def safe_log1p_minus(eps):
    if eps >= 1.0:
        return float("-inf")
    return math.log1p(-eps)


def safe_log(x):
    if x <= 0.0:
        return float("-inf")
    return math.log(x)


# -----------------------------
# IBM-only normalizeCalibration port
# -----------------------------

def normalize_ibm_calibration(a: IBMCalibration) -> CanonicalCalibration:
    qubit_metrics = a.qubit_metrics
    edge_metrics = {edge_key(e[0], e[1]): m for e, m in a.edge_metrics.items()}

    qubits = a.qubits
    edges = list(dict.fromkeys(edge_key(x, y) for x, y in a.edges))

    if qubit_metrics:
        t1_map = {q: m.t1_seconds for q, m in qubit_metrics.items() if m.t1_seconds is not None}
    else:
        t1_map = dict(a.t1_seconds)

    if qubit_metrics:
        t2_map = {q: m.t2_seconds for q, m in qubit_metrics.items() if m.t2_seconds is not None}
    else:
        t2_map = dict(a.t2_seconds)

    fallback_readout_fid = 1.0 - ((a.prob_measu0_prep1 + a.prob_measu1_prep0) / 2.0)

    if qubit_metrics:
        readout_map = {
            q: rf
            for q, m in qubit_metrics.items()
            for rf in [readout_fidelity(m)]
            if rf is not None
        }
    else:
        readout_map = {q: fallback_readout_fid for q in qubits}

    eps1q_map = {}
    for q in qubits:
        metrics = qubit_metrics.get(q, QubitCalibrationMetrics())

        eps1q_map[(q, "ID")] = metric_error(metrics, "ID") if metric_error(metrics, "ID") is not None else a.id_error
        eps1q_map[(q, "RX")] = metric_error(metrics, "RX", "SX", "X") if metric_error(metrics, "RX", "SX", "X") is not None else a.rx_error
        eps1q_map[(q, "SX")] = metric_error(metrics, "SX", "RX", "X") if metric_error(metrics, "SX", "RX", "X") is not None else a.rx_error
        eps1q_map[(q, "X")] = metric_error(metrics, "X", "SX", "RX") if metric_error(metrics, "X", "SX", "RX") is not None else a.pauli_x_error
        eps1q_map[(q, "RZ")] = metric_error(metrics, "RZ") if metric_error(metrics, "RZ") is not None else 0.0
        eps1q_map[(q, "H")] = metric_error(metrics, "H", "SX", "RX", "X") if metric_error(metrics, "H", "SX", "RX", "X") is not None else a.rx_error
        eps1q_map[(q, "Rotate")] = metric_error(metrics, "RX", "SX", "X") if metric_error(metrics, "RX", "SX", "X") is not None else a.rx_error

    eps2q_map = {}
    for edge in edges:
        metrics = edge_metrics.get(edge, EdgeCalibrationMetrics())

        cx_err = edge_metric_error(metrics, "CX", "ECR", "CZ")
        if cx_err is None:
            cx_err = a.cz_error

        cz_err = edge_metric_error(metrics, "CZ", "ECR", "CX")
        if cz_err is None:
            cz_err = a.cz_error

        rzz_err = edge_metric_error(metrics, "RZZ")
        if rzz_err is None:
            rzz_err = a.rzz_error

        eps2q_map[(edge, "CZ")] = cz_err
        eps2q_map[(edge, "RZZ")] = rzz_err
        eps2q_map[(edge, "CX")] = cx_err
        eps2q_map[(edge, "CRotate")] = rzz_err
        eps2q_map[(edge, "SWAP")] = min(1.0, 3.0 * cx_err)

        maybe_ecr = edge_metric_error(metrics, "ECR")
        if maybe_ecr is not None:
            eps2q_map[(edge, "ECR")] = maybe_ecr

    if qubit_metrics:
        measured1q_errors = [
            err
            for metrics in qubit_metrics.values()
            for err in metrics.gate_errors.values()
        ]
    else:
        measured1q_errors = [x for x in [a.id_error, a.rx_error, a.pauli_x_error] if x >= 0.0]

    if edge_metrics:
        measured2q_errors = [
            err
            for metrics in edge_metrics.values()
            for err in metrics.gate_errors.values()
        ]
    else:
        measured2q_errors = [x for x in [a.cz_error, a.rzz_error] if x >= 0.0]

    dur1q_ns = {
        "ID": avg_long_option(metric_duration(qubit_metrics.get(q, QubitCalibrationMetrics()), "ID") for q in qubits) or a.id_length_ns,
        "RX": avg_long_option(metric_duration(qubit_metrics.get(q, QubitCalibrationMetrics()), "RX", "SX", "X") for q in qubits) or a.single_q_gate_length_ns,
        "SX": avg_long_option(metric_duration(qubit_metrics.get(q, QubitCalibrationMetrics()), "SX", "RX", "X") for q in qubits) or a.single_q_gate_length_ns,
        "X": avg_long_option(metric_duration(qubit_metrics.get(q, QubitCalibrationMetrics()), "X", "SX", "RX") for q in qubits) or a.single_q_gate_length_ns,
        "RZ": 0,
        "H": avg_long_option(metric_duration(qubit_metrics.get(q, QubitCalibrationMetrics()), "H", "SX", "RX", "X") for q in qubits) or a.single_q_gate_length_ns,
        "Rotate": avg_long_option(metric_duration(qubit_metrics.get(q, QubitCalibrationMetrics()), "RX", "SX", "X") for q in qubits) or a.single_q_gate_length_ns,
    }

    cx_dur = avg_long_option(edge_metric_duration(edge_metrics.get(e, EdgeCalibrationMetrics()), "CX", "ECR", "CZ") for e in edges) or a.two_q_gate_length_ns
    cz_dur = avg_long_option(edge_metric_duration(edge_metrics.get(e, EdgeCalibrationMetrics()), "CZ", "ECR", "CX") for e in edges) or a.cz_gate_length_ns
    rzz_dur = avg_long_option(edge_metric_duration(edge_metrics.get(e, EdgeCalibrationMetrics()), "RZZ") for e in edges) or a.rzz_gate_length_ns
    ecr_dur = avg_long_option(edge_metric_duration(edge_metrics.get(e, EdgeCalibrationMetrics()), "ECR") for e in edges)

    dur2q_ns = {
        "CZ": cz_dur,
        "RZZ": rzz_dur,
        "CX": cx_dur,
        "CRotate": rzz_dur,
        "SWAP": 3 * cx_dur,
    }

    if ecr_dur is not None:
        dur2q_ns["ECR"] = ecr_dur

    dur_meas_ns = avg_long_option(
        metric_duration(qubit_metrics.get(q, QubitCalibrationMetrics()), "MEASURE")
        for q in qubits
    )

    if dur_meas_ns is None and a.readout_length_ns > 0:
        dur_meas_ns = a.readout_length_ns

    return CanonicalCalibration(
        eps1q=eps1q_map,
        eps2q=eps2q_map,
        eps1q_avg=avg_option(measured1q_errors),
        eps2q_avg=avg_option(measured2q_errors),
        readout_fidelity=readout_map,
        readout_fidelity_avg=avg_option(readout_map.values()) or pct_to_prob(fallback_readout_fid * 100.0),
        t1=t1_map,
        t2=t2_map,
        t1_avg=avg_option(t1_map.values()) or seconds_opt(a.t1_avg_seconds),
        t2_avg=avg_option(t2_map.values()) or seconds_opt(a.t2_avg_seconds),
        dur1q_ns=dur1q_ns,
        dur2q_ns=dur2q_ns,
        dur_meas_ns=dur_meas_ns,
        dur1q_avg_ns=avg_long_option(dur1q_ns.values()),
        dur2q_avg_ns=avg_long_option(dur2q_ns.values()),
        init_survival_per_qubit=None,
        execution_model=ExecutionModel.ParallelByQubitOrEdge,
    )


# -----------------------------
# Backend calibration extraction
# -----------------------------

def backend_name(backend):
    name = getattr(backend, "name", None)
    if callable(name):
        return str(name())
    return str(name)


def seconds_to_ns(x):
    if x is None:
        return None
    try:
        x = float(x)
    except Exception:
        return None
    if math.isfinite(x) and x > 0.0:
        return int(x * 1e9)
    return None


def get_num_qubits(backend):
    n = getattr(backend, "num_qubits", None)
    if n is not None:
        return int(n)

    return int(backend.configuration().num_qubits)


def extract_ibm_calibration_from_backend(backend) -> IBMCalibration:
    target = backend.target
    n = get_num_qubits(backend)

    qubits = list(range(n))
    qubit_metrics = {q: QubitCalibrationMetrics() for q in qubits}
    edge_metrics = {}
    edges = set()

    for q in qubits:
        try:
            qp = backend.qubit_properties(q)
            if qp is not None:
                t1 = getattr(qp, "t1", None)
                t2 = getattr(qp, "t2", None)

                if t1 is not None and math.isfinite(float(t1)) and float(t1) > 0.0:
                    qubit_metrics[q].t1_seconds = float(t1)

                if t2 is not None and math.isfinite(float(t2)) and float(t2) > 0.0:
                    qubit_metrics[q].t2_seconds = float(t2)
        except Exception:
            pass

    for op_name in list(getattr(target, "operation_names", [])):
        try:
            inst_map = target[op_name]
        except Exception:
            continue

        op_upper = upper(op_name)

        try:
            items = inst_map.items()
        except Exception:
            continue

        for qargs, props in items:
            if qargs is None:
                continue

            qargs = tuple(int(q) for q in qargs)

            err = None
            duration_ns = None

            if props is not None:
                raw_err = getattr(props, "error", None)
                if raw_err is not None:
                    try:
                        raw_err = float(raw_err)
                        if math.isfinite(raw_err):
                            err = raw_err
                    except Exception:
                        pass

                duration_ns = seconds_to_ns(getattr(props, "duration", None))

            if len(qargs) == 1:
                q = qargs[0]
                qm = qubit_metrics.setdefault(q, QubitCalibrationMetrics())

                if op_upper == "MEASURE":
                    if err is not None:
                        qm.readout_error = err
                    if duration_ns is not None:
                        qm.gate_durations_ns["MEASURE"] = duration_ns
                else:
                    if err is not None:
                        qm.gate_errors[op_upper] = err
                    if duration_ns is not None:
                        qm.gate_durations_ns[op_upper] = duration_ns

            elif len(qargs) == 2:
                e = edge_key(qargs[0], qargs[1])
                edges.add(e)

                em = edge_metrics.setdefault(e, EdgeCalibrationMetrics())

                if err is not None:
                    em.gate_errors[op_upper] = err

                if duration_ns is not None:
                    em.gate_durations_ns[op_upper] = duration_ns

    t1_values = [(q, m.t1_seconds) for q, m in qubit_metrics.items() if m.t1_seconds is not None]
    t2_values = [(q, m.t2_seconds) for q, m in qubit_metrics.items() if m.t2_seconds is not None]

    readout_errors = [m.readout_error for m in qubit_metrics.values() if m.readout_error is not None]
    avg_readout_error = avg_option(readout_errors) or 0.0

    oneq_errors = [
        err
        for m in qubit_metrics.values()
        for gate, err in m.gate_errors.items()
        if gate in {"ID", "RX", "SX", "X", "RZ", "H"}
    ]

    twoq_errors = [
        err
        for m in edge_metrics.values()
        for gate, err in m.gate_errors.items()
        if gate in {"CX", "ECR", "CZ", "RZZ"}
    ]

    id_errors = [m.gate_errors["ID"] for m in qubit_metrics.values() if "ID" in m.gate_errors]
    rx_like_errors = [
        m.gate_errors[g]
        for m in qubit_metrics.values()
        for g in ["RX", "SX", "X"]
        if g in m.gate_errors
    ]
    x_like_errors = [
        m.gate_errors[g]
        for m in qubit_metrics.values()
        for g in ["X", "SX", "RX"]
        if g in m.gate_errors
    ]
    cz_like_errors = [
        m.gate_errors[g]
        for m in edge_metrics.values()
        for g in ["CZ", "ECR", "CX"]
        if g in m.gate_errors
    ]
    rzz_errors = [
        m.gate_errors["RZZ"]
        for m in edge_metrics.values()
        if "RZZ" in m.gate_errors
    ]

    id_durs = [
        m.gate_durations_ns["ID"]
        for m in qubit_metrics.values()
        if "ID" in m.gate_durations_ns
    ]
    oneq_durs = [
        m.gate_durations_ns[g]
        for m in qubit_metrics.values()
        for g in ["RX", "SX", "X"]
        if g in m.gate_durations_ns
    ]
    twoq_durs = [
        m.gate_durations_ns[g]
        for m in edge_metrics.values()
        for g in ["CX", "ECR", "CZ"]
        if g in m.gate_durations_ns
    ]
    cz_durs = [
        m.gate_durations_ns[g]
        for m in edge_metrics.values()
        for g in ["CZ", "ECR", "CX"]
        if g in m.gate_durations_ns
    ]
    rzz_durs = [
        m.gate_durations_ns["RZZ"]
        for m in edge_metrics.values()
        if "RZZ" in m.gate_durations_ns
    ]
    meas_durs = [
        m.gate_durations_ns["MEASURE"]
        for m in qubit_metrics.values()
        if "MEASURE" in m.gate_durations_ns
    ]

    return IBMCalibration(
        qubits=qubits,
        edges=sorted(edges),
        qubit_metrics=qubit_metrics,
        edge_metrics=edge_metrics,
        t1_seconds=t1_values,
        t2_seconds=t2_values,
        prob_measu0_prep1=avg_readout_error,
        prob_measu1_prep0=avg_readout_error,
        id_error=avg_option(id_errors) or 0.0,
        rx_error=avg_option(rx_like_errors) or avg_option(oneq_errors) or 0.0,
        pauli_x_error=avg_option(x_like_errors) or avg_option(oneq_errors) or 0.0,
        cz_error=avg_option(cz_like_errors) or avg_option(twoq_errors) or 0.0,
        rzz_error=avg_option(rzz_errors) or avg_option(twoq_errors) or 0.0,
        id_length_ns=avg_long_option(id_durs) or 0,
        single_q_gate_length_ns=avg_long_option(oneq_durs) or 0,
        two_q_gate_length_ns=avg_long_option(twoq_durs) or 0,
        cz_gate_length_ns=avg_long_option(cz_durs) or 0,
        rzz_gate_length_ns=avg_long_option(rzz_durs) or 0,
        readout_length_ns=avg_long_option(meas_durs) or 0,
        t1_avg_seconds=avg_option(v for _, v in t1_values) or 0.0,
        t2_avg_seconds=avg_option(v for _, v in t2_values) or 0.0,
    )


# -----------------------------
# Verbatim score port
# -----------------------------

@dataclass
class ScheduleState:
    available_by_qubit_ns: Dict[int, int] = field(default_factory=dict)
    global_gate_available_ns: int = 0

    def qubit_available_ns(self, qubits):
        if not qubits:
            return 0
        return max(self.available_by_qubit_ns.get(q, 0) for q in qubits)

    def schedule(self, qubits, duration_ns, uses_global_gate_resource, execution_model):
        local_start = self.qubit_available_ns(qubits)

        if execution_model == ExecutionModel.GlobalSerialGates:
            global_start = self.global_gate_available_ns
        else:
            global_start = 0

        start = max(local_start, global_start)
        end = start + duration_ns

        updated_available = dict(self.available_by_qubit_ns)
        for q in qubits:
            updated_available[q] = end

        if uses_global_gate_resource and execution_model == ExecutionModel.GlobalSerialGates:
            updated_global = end
        else:
            updated_global = self.global_gate_available_ns

        return ScheduleState(updated_available, updated_global)


@dataclass
class ScheduledOperation:
    qubits: Tuple[int, ...]
    duration_ns: int
    uses_global_gate_resource: bool


def scheduled_operation(op: GateOp, cal: CanonicalCalibration):
    k = op.kind

    if k in {"X", "H", "SX", "RX", "RZ", "RY"}:
        return ScheduledOperation(op.qubits, cal.duration_ns_for(op), True)

    if k == "MEASURE":
        return ScheduledOperation(op.qubits, cal.duration_ns_for(op), False)

    if k in {"CX", "CZ", "SWAP", "CRZ"}:
        return ScheduledOperation(op.qubits, cal.duration_ns_for(op), True)

    return None


def schedule_end_times_sec(ops: List[GateOp], cal: CanonicalCalibration):
    state = ScheduleState()

    for op in ops:
        scheduled = scheduled_operation(op, cal)
        if scheduled is None:
            continue

        state = state.schedule(
            qubits=scheduled.qubits,
            duration_ns=scheduled.duration_ns,
            uses_global_gate_resource=scheduled.uses_global_gate_resource,
            execution_model=cal.execution_model,
        )

    return {q: ns / 1e9 for q, ns in state.available_by_qubit_ns.items()}


def touched_qubits(op: GateOp):
    return list(op.qubits)


def score(compiled: CircuitLike, cal: CanonicalCalibration) -> FidelityEstimate:
    end_times_sec = schedule_end_times_sec(compiled.remaining_gates, cal)

    touched = set()
    for op in compiled.remaining_gates:
        touched.update(touched_qubits(op))

    touched_qubit_count = len(touched)

    if touched_qubit_count > 0:
        width_for_correction = touched_qubit_count
    else:
        width_for_correction = compiled.qubits

    log_p_ops = 0.0

    for op in compiled.remaining_gates:
        k = op.kind

        if k in {"X", "H", "RX", "RZ"}:
            eps = cal.eps_for(op)
            log_p_ops += safe_log1p_minus(eps)

        elif k in {"CX", "CZ", "SWAP", "CRZ"}:
            eps = cal.eps_for(op)
            log_p_ops += safe_log1p_minus(eps)

        elif k == "MEASURE":
            q = op.qubits[0]
            fro = cal.readout_fid_for(q)
            log_p_ops += safe_log(fro)

        else:
            log_p_ops += 0.0

    log_p_decoh = 0.0

    for q, t_sec in end_times_sec.items():
        t2_sec = cal.t2_for(q)
        if t2_sec is not None and t2_sec > 0.0:
            log_p_decoh += -t_sec / t2_sec

    log_p_total = log_p_ops + log_p_decoh
    p_total = math.exp(log_p_total) if log_p_total > float("-inf") else 0.0

    est = FidelityEstimate(
        log_p_ops=log_p_ops,
        log_p_decoh=log_p_decoh,
        log_p_total=log_p_total,
        p_total=p_total,
        per_qubit_end_time_sec=end_times_sec,
    )

    if width_for_correction > 20:
        fcoeff = 0.3
    elif width_for_correction > 10:
        fcoeff = 0.7
    else:
        fcoeff = 1.0

    return FidelityEstimate(
        log_p_ops=est.log_p_ops,
        log_p_decoh=est.log_p_decoh,
        log_p_total=est.log_p_total + math.log(fcoeff),
        p_total=est.p_total * fcoeff,
        per_qubit_end_time_sec=est.per_qubit_end_time_sec,
    )


# -----------------------------
# Qiskit circuit helpers
# -----------------------------

def make_ghz(n):
    qc = QuantumCircuit(n, n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    qc.measure(range(n), range(n))
    return qc


def qiskit_to_gate_ops(circuit):
    ops = []

    for item in circuit.data:
        operation = item.operation
        qargs = item.qubits
        name = operation.name.lower()

        qids = tuple(circuit.find_bit(q).index for q in qargs)

        if name in {"barrier", "delay"}:
            continue

        if name == "x":
            ops.append(GateOp("X", qids))
        elif name == "h":
            ops.append(GateOp("H", qids))
        elif name == "sx":
            ops.append(GateOp("SX", qids))
        elif name == "rx":
            ops.append(GateOp("RX", qids))
        elif name == "ry":
            ops.append(GateOp("RY", qids))
        elif name == "rz":
            ops.append(GateOp("RZ", qids))
        elif name == "measure":
            ops.append(GateOp("MEASURE", qids))
        elif name == "cx":
            ops.append(GateOp("CX", qids))
        elif name == "cz":
            ops.append(GateOp("CZ", qids))
        elif name == "swap":
            ops.append(GateOp("SWAP", qids))
        elif name == "crz":
            ops.append(GateOp("CRZ", qids))
        elif name == "ecr":
            # Kept as ECR, but the verbatim Scala score does not count/schedule ECR.
            ops.append(GateOp("ECR", qids))
        else:
            # NamedGate-equivalent; touched qubits are preserved but score ignores it.
            ops.append(GateOp(name.upper(), qids))

    return ops


def ghz_support_fidelity(counts, n, shots):
    cleaned = {}
    for key, value in counts.items():
        k = key.replace(" ", "")
        cleaned[k] = cleaned.get(k, 0) + value

    zero = "0" * n
    one = "1" * n

    p_zero = cleaned.get(zero, 0) / shots
    p_one = cleaned.get(one, 0) / shots

    return p_zero + p_one, p_zero, p_one


def ignored_ops_by_verbatim_estimator(tqc):
    counted = {"x", "h", "rx", "rz", "cx", "cz", "swap", "crz", "measure"}
    ops = dict(tqc.count_ops())
    return {k: int(v) for k, v in ops.items() if k.lower() not in counted}


def run_backend_comparison(device_label, backend):
    ibm_cal = extract_ibm_calibration_from_backend(backend)
    cal = normalize_ibm_calibration(ibm_cal)

    simulator = AerSimulator.from_backend(backend)
    simulator.set_options(seed_simulator=SEED_SIMULATOR)

    rows = []

    for n in range(MIN_GHZ, MAX_GHZ + 1):
        qc = make_ghz(n)

        tqc = transpile(
            qc,
            backend=backend,
            optimization_level=OPTIMIZATION_LEVEL,
            seed_transpiler=SEED_TRANSPILER,
        )

        compiled = CircuitLike(
            qubits=tqc.num_qubits,
            remaining_gates=qiskit_to_gate_ops(tqc),
        )

        est = score(compiled, cal)

        result = simulator.run(tqc, shots=SHOTS).result()
        counts = result.get_counts()

        fake_fidelity, p_zero, p_one = ghz_support_fidelity(counts, n, SHOTS)

        rows.append({
            "device": device_label,
            "backend_name": backend_name(backend),
            "ghz_qubits": n,
            "shots": SHOTS,

            "scala_port_estimated_fidelity": est.p_total,
            "scala_port_log_p_ops": est.log_p_ops,
            "scala_port_log_p_decoh": est.log_p_decoh,
            "scala_port_log_p_total": est.log_p_total,

            "fake_backend_support_fidelity": fake_fidelity,
            "fake_backend_p_zero": p_zero,
            "fake_backend_p_one": p_one,

            "transpiled_depth": tqc.depth(),
            "transpiled_ops": dict(tqc.count_ops()),
            "ignored_ops_by_verbatim_estimator": ignored_ops_by_verbatim_estimator(tqc),
            "per_qubit_end_time_sec": est.per_qubit_end_time_sec,
            "status": "ok",
            "error": "",
        })

    return rows

In [ ]:
all_rows = []

for device_label, class_name in DEVICE_CLASSES.items():
    print(f"\nLoading {device_label} using {class_name}")

    try:
        BackendClass = getattr(fp, class_name)
        backend = BackendClass()

        rows = run_backend_comparison(device_label, backend)
        all_rows.extend(rows)

        for r in rows:
            print(
                f"  GHZ({r['ghz_qubits']}): "
                f"est={r['scala_port_estimated_fidelity']:.6f}, "
                f"fake={r['fake_backend_support_fidelity']:.6f}, "
                f"depth={r['transpiled_depth']}, "
                f"ignored={r['ignored_ops_by_verbatim_estimator']}"
            )

    except Exception as e:
        print(f"  failed: {repr(e)}")
        all_rows.append({
            "device": device_label,
            "backend_name": None,
            "ghz_qubits": None,
            "shots": SHOTS,
            "scala_port_estimated_fidelity": None,
            "fake_backend_support_fidelity": None,
            "status": "failed",
            "error": repr(e),
        })


df = pd.DataFrame(all_rows)
df


Loading ibm_boston using FakeBoston
  GHZ(2): est=0.983915, fake=0.990000, depth=7, ignored={'sx': 3}
  GHZ(3): est=0.973506, fake=0.990000, depth=10, ignored={'sx': 5}
  GHZ(4): est=0.965379, fake=0.995000, depth=13, ignored={'sx': 7}
  GHZ(5): est=0.952899, fake=0.975000, depth=16, ignored={'sx': 9}
  GHZ(6): est=0.940090, fake=0.970000, depth=19, ignored={'sx': 11}
  GHZ(7): est=0.928419, fake=0.970000, depth=22, ignored={'sx': 13}
  GHZ(8): est=0.912228, fake=0.970000, depth=25, ignored={'sx': 15}
  GHZ(9): est=0.896841, fake=0.960000, depth=28, ignored={'sx': 17}
  GHZ(10): est=0.877908, fake=0.960000, depth=31, ignored={'sx': 19}

Loading ibm_marrakesh using FakeMarrakesh
  GHZ(2): est=0.970465, fake=0.985000, depth=7, ignored={'sx': 3}
  GHZ(3): est=0.945585, fake=0.980000, depth=10, ignored={'sx': 5}
  GHZ(4): est=0.919863, fake=0.985000, depth=13, ignored={'sx': 7}
  GHZ(5): est=0.783235, fake=0.955000, depth=16, ignored={'sx': 9}
  GHZ(6): est=0.760498, fake=0.970000, depth=

,device,backend_name,ghz_qubits,shots,scala_port_estimated_fidelity,scala_port_log_p_ops,scala_port_log_p_decoh,scala_port_log_p_total,fake_backend_support_fidelity,fake_backend_p_zero,fake_backend_p_one,transpiled_depth,transpiled_ops,ignored_ops_by_verbatim_estimator,per_qubit_end_time_sec,status,error
0,ibm_boston,fake_boston,2,200,0.983915,-0.004339,-0.011876,-0.016215,0.990,0.520,0.470,7,"{'rz': 5, 'sx': 3, 'measure': 2, 'cz': 1}",{'sx': 3},"{14: 2.3e-06, 15: 2.332e-06}",ok,
1,ibm_boston,fake_boston,3,200,0.973506,-0.008746,-0.018105,-0.026851,0.990,0.520,0.470,10,"{'rz': 8, 'sx': 5, 'measure': 3, 'cz': 2}",{'sx': 5},"{22: 2.432e-06, 23: 2.4e-06, 24: 2.3e-06}",ok,
2,ibm_boston,fake_boston,4,200,0.965379,-0.012728,-0.022507,-0.035235,0.995,0.525,0.470,13,"{'rz': 11, 'sx': 7, 'measure': 4, 'cz': 3}",{'sx': 7},"{14: 2.3e-06, 15: 2.4e-06, 19: 2.5e-06, 35: 2....",ok,
3,ibm_boston,fake_boston,5,200,0.952899,-0.018784,-0.029463,-0.048247,0.975,0.515,0.460,16,"{'rz': 14, 'sx': 9, 'measure': 5, 'cz': 4}",{'sx': 9},"{3: 2.4e-06, 4: 2.3e-06, 16: 2.5e-06, 22: 2.63...",ok,
4,ibm_boston,fake_boston,6,200,0.940090,-0.025200,-0.036579,-0.061779,0.970,0.505,0.465,19,"{'rz': 17, 'sx': 11, 'measure': 6, 'cz': 5}",{'sx': 11},"{3: 2.3e-06, 16: 2.4e-06, 20: 2.732e-06, 21: 2...",ok,
5,ibm_boston,fake_boston,7,200,0.928419,-0.030285,-0.043987,-0.074272,0.970,0.510,0.460,22,"{'rz': 20, 'sx': 13, 'measure': 7, 'cz': 6}",{'sx': 13},"{3: 2.4e-06, 4: 2.3e-06, 16: 2.5e-06, 20: 2.83...",ok,
6,ibm_boston,fake_boston,8,200,0.912228,-0.034247,-0.057619,-0.091866,0.970,0.505,0.465,25,"{'rz': 23, 'sx': 15, 'measure': 8, 'cz': 7}",{'sx': 15},"{14: 2.3e-06, 15: 2.4e-06, 19: 2.5e-06, 31: 2....",ok,
7,ibm_boston,fake_boston,9,200,0.896841,-0.039260,-0.069616,-0.108876,0.960,0.495,0.465,28,"{'rz': 26, 'sx': 17, 'measure': 9, 'cz': 8}",{'sx': 17},"{14: 2.3e-06, 15: 2.4e-06, 18: 3.032e-06, 19: ...",ok,
8,ibm_boston,fake_boston,10,200,0.877908,-0.045927,-0.084286,-0.130213,0.960,0.495,0.465,31,"{'rz': 29, 'sx': 19, 'measure': 10, 'cz': 9}",{'sx': 19},"{21: 2.5e-06, 22: 2.4e-06, 23: 2.3e-06, 36: 2....",ok,
9,ibm_marrakesh,fake_marrakesh,2,200,0.970465,-0.010014,-0.019966,-0.029980,0.985,0.520,0.465,7,"{'rz': 5, 'sx': 3, 'measure': 2, 'cz': 1}",{'sx': 3},"{5: 2.724e-06, 6: 2.688e-06}",ok,


In [ ]:
import math
import pandas as pd
from getpass import getpass

from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler


SHOTS = 200
GHZ_SIZES = [8, 10]
OPTIMIZATION_LEVEL = 3
SEED_TRANSPILER = 1234

DEVICE_NAME_CANDIDATES = {
    "ibm_boston": ["ibm_boston"],
    "ibm_marrakesh": ["ibm_marrakesh"],
    "ibm_kingston": ["ibm_kingston"],
    "ibm_pittsburg": ["ibm_pittsburg", "ibm_pittsburgh"],
    "ibm_fez": ["ibm_fez"],
}

IBM_TOKEN = ""
IBM_INSTANCE = ""

service = QiskitRuntimeService(
    channel="ibm_cloud",
    token=IBM_TOKEN,
    instance=IBM_INSTANCE,
)


def backend_name(backend):
    name = getattr(backend, "name", None)
    return name() if callable(name) else str(name)


def make_ghz(n):
    qc = QuantumCircuit(n)
    qc.h(0)

    for i in range(n - 1):
        qc.cx(i, i + 1)

    qc.measure_all()
    return qc


def resolve_backend(label, candidates):
    for name in candidates:
        try:
            backend = service.backend(name=name)
            return backend
        except Exception:
            pass

    return None


def get_num_qubits(backend):
    n = getattr(backend, "num_qubits", None)
    if n is not None:
        return int(n)
    return int(backend.configuration().num_qubits)


def ghz_observed_fidelity_from_counts(counts, n, shots):
    cleaned = {}

    for key, value in counts.items():
        k = key.replace(" ", "")
        cleaned[k] = cleaned.get(k, 0) + value

    zero = "0" * n
    one = "1" * n

    c_zero = cleaned.get(zero, 0)
    c_one = cleaned.get(one, 0)

    p_zero = c_zero / shots
    p_one = c_one / shots
    fidelity = p_zero + p_one

    return fidelity, p_zero, p_one, c_zero, c_one


def get_counts_from_pub_result(pub_result):
    if hasattr(pub_result.data, "meas"):
        return pub_result.data.meas.get_counts()

    for field_name in dir(pub_result.data):
        if field_name.startswith("_"):
            continue
        value = getattr(pub_result.data, field_name)
        if hasattr(value, "get_counts"):
            return value.get_counts()

    raise RuntimeError("Could not find a BitArray with get_counts() in the Sampler result.")


rows = []

for label, candidates in DEVICE_NAME_CANDIDATES.items():
    print(f"\nResolving {label}: candidates={candidates}")

    backend = resolve_backend(label, candidates)

    if backend is None:
        print(f"  Could not find an accessible real backend for {label}. Skipping.")
        rows.append({
            "device_label": label,
            "backend_name": None,
            "ghz_qubits": None,
            "shots": SHOTS,
            "observed_fidelity": None,
            "p_zero": None,
            "p_one": None,
            "counts_zero": None,
            "counts_one": None,
            "job_id": None,
            "status": "backend_not_accessible",
            "error": "No candidate backend name resolved.",
        })
        continue

    real_name = backend_name(backend)
    num_qubits = get_num_qubits(backend)

    print(f"  Using backend: {real_name}, qubits={num_qubits}")

    circuits = []
    circuit_sizes = []

    pm = generate_preset_pass_manager(
        backend=backend,
        optimization_level=OPTIMIZATION_LEVEL,
        seed_transpiler=SEED_TRANSPILER,
    )

    for n in GHZ_SIZES:
        if num_qubits < n:
            print(f"  Skipping GHZ({n}): backend has only {num_qubits} qubits.")
            rows.append({
                "device_label": label,
                "backend_name": real_name,
                "ghz_qubits": n,
                "shots": SHOTS,
                "observed_fidelity": None,
                "p_zero": None,
                "p_one": None,
                "counts_zero": None,
                "counts_one": None,
                "job_id": None,
                "status": "too_few_qubits",
                "error": "",
            })
            continue

        logical = make_ghz(n)
        isa_circuit = pm.run(logical)

        circuits.append(isa_circuit)
        circuit_sizes.append(n)

        print(
            f"  Prepared GHZ({n}): "
            f"depth={isa_circuit.depth()}, ops={dict(isa_circuit.count_ops())}"
        )

    if not circuits:
        continue

    try:
        sampler = Sampler(mode=backend)
        job = sampler.run(circuits, shots=SHOTS)

        job_id_attr = getattr(job, "job_id", None)
        job_id = job_id_attr() if callable(job_id_attr) else str(job_id_attr)

        print(f"  Submitted job: {job_id}")
        print("  Waiting for result...")

        result = job.result()

        for i, n in enumerate(circuit_sizes):
            pub_result = result[i]
            counts = get_counts_from_pub_result(pub_result)

            fidelity, p_zero, p_one, c_zero, c_one = ghz_observed_fidelity_from_counts(
                counts=counts,
                n=n,
                shots=SHOTS,
            )

            rows.append({
                "device_label": label,
                "backend_name": real_name,
                "ghz_qubits": n,
                "shots": SHOTS,
                "observed_fidelity": fidelity,
                "p_zero": p_zero,
                "p_one": p_one,
                "counts_zero": c_zero,
                "counts_one": c_one,
                "job_id": job_id,
                "status": "ok",
                "error": "",
            })

            print(
                f"  GHZ({n}) observed fidelity={fidelity:.4f}, "
                f"p0={p_zero:.4f}, p1={p_one:.4f}, "
                f"counts_zero={c_zero}, counts_one={c_one}"
            )

    except Exception as e:
        print(f"  Job failed for {real_name}: {repr(e)}")

        for n in circuit_sizes:
            rows.append({
                "device_label": label,
                "backend_name": real_name,
                "ghz_qubits": n,
                "shots": SHOTS,
                "observed_fidelity": None,
                "p_zero": None,
                "p_one": None,
                "counts_zero": None,
                "counts_one": None,
                "job_id": None,
                "status": "failed",
                "error": repr(e),
            })


df_real = pd.DataFrame(rows)
df_real

qiskit_runtime_service._discover_account:WARNING:2026-07-22 20:48:17,985: Loading account with the given token. A saved account will not be used.



Resolving ibm_boston: candidates=['ibm_boston']
  Using backend: ibm_boston, qubits=156
  Prepared GHZ(8): depth=25, ops={'rz': 23, 'sx': 15, 'measure': 8, 'cz': 7, 'barrier': 1}
  Prepared GHZ(10): depth=31, ops={'rz': 29, 'sx': 19, 'measure': 10, 'cz': 9, 'barrier': 1}
  Submitted job: d9gip6bsbqfc73eon0eg
  Waiting for result...
  GHZ(8) observed fidelity=0.9350, p0=0.4800, p1=0.4550, counts_zero=96, counts_one=91
  GHZ(10) observed fidelity=0.9100, p0=0.5000, p1=0.4100, counts_zero=100, counts_one=82

Resolving ibm_marrakesh: candidates=['ibm_marrakesh']
  Using backend: ibm_marrakesh, qubits=156
  Prepared GHZ(8): depth=25, ops={'rz': 23, 'sx': 15, 'measure': 8, 'cz': 7, 'barrier': 1}
  Prepared GHZ(10): depth=31, ops={'rz': 29, 'sx': 19, 'measure': 10, 'cz': 9, 'barrier': 1}
  Submitted job: d9givbbsbqfc73eon9lg
  Waiting for result...
  GHZ(8) observed fidelity=0.9200, p0=0.5050, p1=0.4150, counts_zero=101, counts_one=83
  GHZ(10) observed fidelity=0.8450, p0=0.4800, p1=0.3650,

,device_label,backend_name,ghz_qubits,shots,observed_fidelity,p_zero,p_one,counts_zero,counts_one,job_id,status,error
0,ibm_boston,ibm_boston,8,200,0.935,0.480,0.455,96,91,d9gip6bsbqfc73eon0eg,ok,
1,ibm_boston,ibm_boston,10,200,0.910,0.500,0.410,100,82,d9gip6bsbqfc73eon0eg,ok,
2,ibm_marrakesh,ibm_marrakesh,8,200,0.920,0.505,0.415,101,83,d9givbbsbqfc73eon9lg,ok,
3,ibm_marrakesh,ibm_marrakesh,10,200,0.845,0.480,0.365,96,73,d9givbbsbqfc73eon9lg,ok,
4,ibm_kingston,ibm_kingston,8,200,0.905,0.505,0.400,101,80,d9gj3logk0ls73f1l2i0,ok,
5,ibm_kingston,ibm_kingston,10,200,0.880,0.480,0.400,96,80,d9gj3logk0ls73f1l2i0,ok,
6,ibm_pittsburg,ibm_pittsburgh,8,200,0.905,0.475,0.430,95,86,d9gj45l0k0jc738grj90,ok,
7,ibm_pittsburg,ibm_pittsburgh,10,200,0.890,0.460,0.430,92,86,d9gj45l0k0jc738grj90,ok,
8,ibm_fez,ibm_fez,8,200,0.880,0.545,0.335,109,67,d9gk0qbsbqfc73eooej0,ok,
9,ibm_fez,ibm_fez,10,200,0.885,0.470,0.415,94,83,d9gk0qbsbqfc73eooej0,ok,


In [3]:
import math
import numpy as np
import pandas as pd
from getpass import getpass

from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

SHOTS = 200
GHZ_SIZES = [8, 10]
OPTIMIZATION_LEVEL = 3
SEED_TRANSPILER = 1234
NUM_THETA_STEPS = 8

DEVICE_NAME_CANDIDATES = {
    "ibm_boston": ["ibm_boston"],
    "ibm_marrakesh": ["ibm_marrakesh"],
    "ibm_kingston": ["ibm_kingston"],
    "ibm_pittsburg": ["ibm_pittsburg", "ibm_pittsburgh"],
    "ibm_fez": ["ibm_fez"],
}

IBM_TOKEN = ""
IBM_INSTANCE = ""

service = QiskitRuntimeService(
    channel="ibm_cloud",
    token=IBM_TOKEN,
    instance=IBM_INSTANCE,
)


def backend_name(backend):
    name = getattr(backend, "name", None)
    return name() if callable(name) else str(name)


def get_num_qubits(backend):
    n = getattr(backend, "num_qubits", None)
    if n is not None:
        return int(n)
    return int(backend.configuration().num_qubits)


def resolve_backend(label, candidates):
    for name in candidates:
        try:
            backend = service.backend(name=name)
            return backend
        except Exception:
            pass
    return None


def make_ghz_population(n):
    """Circuit measured in standard Z-basis to extract state population."""
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    qc.measure_all()
    return qc


def make_ghz_parity_step(n, theta):
    """Circuit rotated into the X-Y plane to detect phase coherence."""
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)

    for i in range(n):
        qc.rz(theta, i)
        qc.h(i)

    qc.measure_all()
    return qc


def get_counts_from_pub_result(pub_result):
    if hasattr(pub_result.data, "meas"):
        return pub_result.data.meas.get_counts()
    for field_name in dir(pub_result.data):
        if field_name.startswith("_"):
            continue
        value = getattr(pub_result.data, field_name)
        if hasattr(value, "get_counts"):
            return value.get_counts()
    raise RuntimeError("Could not find a BitArray with get_counts() in the Sampler result.")


def compute_parity_from_counts(counts):
    """Calculates <Z_1 Z_2 ... Z_n> parity expectation value."""
    total_shots = sum(counts.values())
    parity_sum = 0
    for bitstring, count in counts.items():
        clean_str = bitstring.replace(" ", "")
        ones_count = clean_str.count('1')
        sign = 1 if ones_count % 2 == 0 else -1
        parity_sum += sign * count
    return parity_sum / total_shots


def analyze_ghz_results(pop_counts, parity_counts_list, theta_list, n):
    """Extracts population, fits coherence amplitude, and outputs true quantum fidelity."""

    clean_pop = {k.replace(" ", ""): v for k, v in pop_counts.items()}
    total_pop_shots = sum(clean_pop.values())

    c_zero = clean_pop.get("0" * n, 0)
    c_one = clean_pop.get("1" * n, 0)

    p_zero = c_zero / total_pop_shots
    p_one = c_one / total_pop_shots
    population = p_zero + p_one
    parities = [compute_parity_from_counts(cts) for cts in parity_counts_list]

    cos_projections = [p * math.cos(n * t) for p, t in zip(parities, theta_list)]
    sin_projections = [p * math.sin(n * t) for p, t in zip(parities, theta_list)]

    amplitude = 2 * math.sqrt(np.mean(cos_projections)**2 + np.mean(sin_projections)**2)

    true_fidelity = 0.5 * population + 0.5 * amplitude

    return true_fidelity, population, amplitude, p_zero, p_one


rows = []

for label, candidates in DEVICE_NAME_CANDIDATES.items():
    print(f"\nResolving {label}: candidates={candidates}")
    backend = resolve_backend(label, candidates)

    if backend is None:
        print(f"  Could not find an accessible real backend for {label}. Skipping.")
        rows.append({
            "device_label": label, "backend_name": None, "ghz_qubits": None,
            "true_fidelity": None, "population": None, "coherence_amplitude": None,
            "status": "backend_not_accessible"
        })
        continue

    real_name = backend_name(backend)
    num_qubits = get_num_qubits(backend)
    print(f"  Using backend: {real_name}, qubits={num_qubits}")

    pm = generate_preset_pass_manager(
        backend=backend,
        optimization_level=OPTIMIZATION_LEVEL,
        seed_transpiler=SEED_TRANSPILER,
    )

    for n in GHZ_SIZES:
        if num_qubits < n:
            print(f"  Skipping GHZ({n}): backend has only {num_qubits} qubits.")
            continue

        logical_circuits = [make_ghz_population(n)]
        theta_steps = [2 * math.pi * k / (n * NUM_THETA_STEPS) for k in range(NUM_THETA_STEPS)]

        for theta in theta_steps:
            logical_circuits.append(make_ghz_parity_step(n, theta))

        isa_circuits = [pm.run(qc) for qc in logical_circuits]
        print(f"  Prepared batch for GHZ({n}): {len(isa_circuits)} circuits submitted.")

        try:
            sampler = Sampler(mode=backend)
            job = sampler.run(isa_circuits, shots=SHOTS)
            job_id = getattr(job, "job_id", lambda: str(job))()
            print(f"  Submitted job: {job_id}. Waiting for hardware evaluation...")

            result = job.result()

            pop_counts = get_counts_from_pub_result(result[0])
            parity_counts_list = [get_counts_from_pub_result(result[i]) for i in range(1, len(isa_circuits))]

            true_fid, pop, amp, p0, p1 = analyze_ghz_results(
                pop_counts, parity_counts_list, theta_steps, n
            )

            rows.append({
                "device_label": label, "backend_name": real_name, "ghz_qubits": n,
                "true_fidelity": true_fid, "population": pop, "coherence_amplitude": amp,
                "p_zero": p0, "p_one": p1, "job_id": job_id, "status": "ok"
            })
            print(f"  [RESULT] GHZ({n}) -> True Fidelity: {true_fid:.4f} | Pop: {pop:.4f} | Coh-Amp: {amp:.4f}")

        except Exception as e:
            print(f"  Execution failed for {real_name}: {repr(e)}")
            rows.append({
                "device_label": label, "backend_name": real_name, "ghz_qubits": n,
                "status": "failed", "error": repr(e)
            })

df_real = pd.DataFrame(rows)
df_real


qiskit_runtime_service._discover_account:WARNING:2026-07-30 05:20:03,614: Loading account with the given token. A saved account will not be used.



Resolving ibm_boston: candidates=['ibm_boston']
  Using backend: ibm_boston, qubits=156
  Prepared batch for GHZ(8): 9 circuits submitted.
  Submitted job: d9ldu22br2fc73e8iea0. Waiting for hardware evaluation...
  [RESULT] GHZ(8) -> True Fidelity: 0.9017 | Pop: 0.9450 | Coh-Amp: 0.8584
  Prepared batch for GHZ(10): 9 circuits submitted.
  Submitted job: d9le0p3hdfks73ckruu0. Waiting for hardware evaluation...
  [RESULT] GHZ(10) -> True Fidelity: 0.9126 | Pop: 0.9400 | Coh-Amp: 0.8853

Resolving ibm_marrakesh: candidates=['ibm_marrakesh']
  Using backend: ibm_marrakesh, qubits=156
  Prepared batch for GHZ(8): 9 circuits submitted.
  Submitted job: d9le1f2br2fc73e8iijg. Waiting for hardware evaluation...
  [RESULT] GHZ(8) -> True Fidelity: 0.8348 | Pop: 0.8750 | Coh-Amp: 0.7947
  Prepared batch for GHZ(10): 9 circuits submitted.
  Submitted job: d9le1nqbr2fc73e8iitg. Waiting for hardware evaluation...
  [RESULT] GHZ(10) -> True Fidelity: 0.7839 | Pop: 0.8300 | Coh-Amp: 0.7378

Resolvin

,device_label,backend_name,ghz_qubits,true_fidelity,population,coherence_amplitude,p_zero,p_one,job_id,status
0,ibm_boston,ibm_boston,8,0.901704,0.945,0.858408,0.420,0.525,d9ldu22br2fc73e8iea0,ok
1,ibm_boston,ibm_boston,10,0.912639,0.940,0.885277,0.515,0.425,d9le0p3hdfks73ckruu0,ok
2,ibm_marrakesh,ibm_marrakesh,8,0.834839,0.875,0.794677,0.495,0.380,d9le1f2br2fc73e8iijg,ok
3,ibm_marrakesh,ibm_marrakesh,10,0.783911,0.830,0.737821,0.525,0.305,d9le1nqbr2fc73e8iitg,ok
4,ibm_kingston,ibm_kingston,8,0.821736,0.875,0.768471,0.435,0.440,d9le1prhdfks73cks050,ok
5,ibm_kingston,ibm_kingston,10,0.804588,0.860,0.749176,0.400,0.460,d9le20qbr2fc73e8ij7g,ok
6,ibm_pittsburg,ibm_pittsburgh,8,0.868102,0.920,0.816204,0.455,0.465,d9le2mrhdfks73cks16g,ok
7,ibm_pittsburg,ibm_pittsburgh,10,0.797353,0.845,0.749706,0.395,0.450,d9lehngii2cc73ehd6ng,ok
8,ibm_fez,ibm_fez,8,0.825391,0.860,0.790781,0.405,0.455,d9lehpabr2fc73e8j6tg,ok
9,ibm_fez,ibm_fez,10,0.768055,0.815,0.721109,0.440,0.375,d9lel4rjf64c739je77g,ok
